# Companion notebook (ខ្មែរ) — `ch07/01_main-chapter-code/load-finetuned-model.ipynb`

> **ប្រភព និងអាជ្ញាបណ្ណ** — កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង
> [`ch07/01_main-chapter-code/load-finetuned-model.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/load-finetuned-model.ipynb)
> នៃឃ្លាំង [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)
> (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។ Comments ក្នុងកូដត្រូវបានបកប្រែជាខ្មែរដោយសហគមន៍។
> Notebook នេះ **មិនមានអត្ថបទសៀវភៅទេ** (អត្ថបទសៀវភៅជាកម្មសិទ្ធិ Manning ដកចេញពីអាជ្ញាបណ្ណបើកចំហ) —
> សម្រាប់ការពន្យល់សូមមើលសៀវភៅផ្លូវការ ឬ notebook ដើម។

In [ ]:
# ការតភ្ជាប់ឯកសារជំនួយ — ឱ្យដំណើរការភ្ជាប់ដូច notebook ដើមរបស់ upstream
# (ចង្អុល sys.path និងថតធ្វើការទៅកាន់ថតកូដក្នុងឃ្លាំងកសារខ្មែរនេះ)
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "កូដ")):
    _p = os.path.dirname(_p)
_UP = os.path.join(_p, "កូដ", "ch07/01_main-chapter-code")
sys.path.insert(0, _UP)
os.chdir(_UP)


In [ ]:
from importlib.metadata import version

pkgs = [
    "tiktoken",  # Tokenizer (កម្មវិធីបំបែក token)
    "torch",  # បណ្ណាល័យ deep learning
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

finetuned_model_path = Path("gpt2-medium355M-sft.pth")
if not finetuned_model_path.exists():
    print(
        f"Could not find '{finetuned_model_path}'.\n"
        "Run the `ch07.ipynb` notebook to finetune and save the finetuned model."
    )

In [ ]:
from previous_chapters import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,  # ទំហំ vocabulary
    "context_length": 1024,  # ប្រវែង context
    "drop_rate": 0.0,  # អត្រា dropout
    "qkv_bias": True  # bias សម្រាប់ query-key-value
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
model = GPTModel(BASE_CONFIG)

In [ ]:
import torch

model.load_state_dict(torch.load(
    "gpt2-medium355M-sft.pth",
    map_location=torch.device("cpu"),
    weights_only=True
))
model.eval();

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
prompt = """Below is an instruction that describes a task. Write a response 
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

def extract_response(response_text, input_text):
    return response_text[len(input_text):].replace("### Response:", "").strip()

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)
response = extract_response(response, prompt)
print(response)

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ